# Final Project Walkthrough: S&P 500 MLOps Pipeline

This notebook is an executable support artefact for the S&P 500 market-direction MLOps project. It mirrors the implemented pipeline stages and keeps the modelling scope deliberately simple: a Logistic Regression baseline and a Random Forest challenger trained on Yahoo Finance OHLCV data for `^GSPC`.

The goal is not to build a trading strategy. The goal is to demonstrate a working, reproducible MLOps workflow: ingestion, quality gates, cleaning, feature engineering, validation contracts, drift checks, chronological splitting, model training, model comparison, test evaluation, artefact persistence, explainability and a serving-style prediction example.


## 1. Pipeline Map and Experiment Design

The project pipeline is organised as the following stages:

1. `data_ingestion`: download or load S&P 500 OHLCV data.
2. `data_quality`: validate raw schema, dates, prices and volume.
3. `data_cleaning`: standardise types, ordering and duplicate handling.
4. `data_feat_engineering`: create market features and the next-day binary target.
5. `data_expectations`: run explicit data contracts for raw and feature datasets.
6. `data_drift`: compare older reference data with the most recent period.
7. `data_split`: create chronological train, validation and test partitions.
8. `model_train`: train Logistic Regression as the baseline candidate.
9. `model_train_challenger`: train Random Forest as the challenger candidate.
10. `model_selection`: choose the candidate champion using validation metrics.
11. `model_predict` and `model_predict_challenger`: evaluate held-out test predictions.
12. `model_explainability`: produce interpretable feature-importance artefacts.
13. `serving`: prepare a single-row prediction payload compatible with online inference.

The modelling target is `target_next_day_up`: whether the next available S&P 500 close is higher than the current close. Splits are chronological to reduce look-ahead bias. Validation is used for model selection; the test set is reserved for final reporting.


## 2. Imports, Reproducibility and Configuration

Run this notebook from the repository root. The configuration below keeps the dates, ticker, feature list and artefact locations explicit so the notebook can be used as a transparent support pipeline for the MLOps project.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import pickle
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        "Run this notebook from the project root or from the notebooks directory."
    )

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from sp500_mlops_pipeline.pipelines.data_cleaning.nodes import clean_raw_market_data
from sp500_mlops_pipeline.pipelines.data_drift.nodes import (
    create_data_drift_report,
    select_numeric_drift_features,
    split_reference_current_data,
)
from sp500_mlops_pipeline.pipelines.data_expectations.nodes import (
    create_feature_validation_html_report,
    create_feature_validation_report,
    create_raw_validation_html_report,
    create_raw_validation_report,
    enforce_validation_contracts,
    validate_feature_data_expectations,
    validate_raw_market_data_expectations,
)
from sp500_mlops_pipeline.pipelines.data_feat_engineering.nodes import (
    FEATURE_PRICE_COLUMN,
    MARKET_FEATURE_COLUMNS,
    TARGET_COLUMN,
    create_market_feature_dataset,
)
from sp500_mlops_pipeline.pipelines.data_quality.nodes import (
    REQUIRED_COLUMNS,
    validate_raw_market_data,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TICKER = "^GSPC"
START_DATE = "2001-01-01"
TRAIN_END = "2023-12-31"
VALIDATION_START = "2024-01-01"
VALIDATION_END = "2025-12-31"
TEST_START = "2026-01-01"

DOWNLOAD_FROM_YAHOO = True
ALLOW_CACHED_RAW_FALLBACK = True

RAW_DATA_PATH = PROJECT_ROOT / "data/01_raw/sp500_yahoo_finance_raw.csv"
CLEAN_DATA_PATH = PROJECT_ROOT / "data/02_intermediate/sp500_clean_market_data.csv"
FEATURE_DATA_PATH = PROJECT_ROOT / "data/04_feature/sp500_feature_data.csv"
MODEL_DIR = PROJECT_ROOT / "data/06_models"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "data/07_model_output"
REPORTING_DIR = PROJECT_ROOT / "data/08_reporting"
EXPECTATIONS_DIR = REPORTING_DIR / "great_expectations"
DRIFT_DIR = REPORTING_DIR / "drift"
YFINANCE_CACHE_DIR = REPORTING_DIR / "yfinance_cache"

for directory in [RAW_DATA_PATH.parent, CLEAN_DATA_PATH.parent, FEATURE_DATA_PATH.parent, MODEL_DIR, MODEL_OUTPUT_DIR, EXPECTATIONS_DIR, DRIFT_DIR, YFINANCE_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

yf.set_tz_cache_location(str(YFINANCE_CACHE_DIR))

np.random.seed(RANDOM_STATE)

pipeline_names = [
    "data_quality",
    "data_cleaning",
    "data_feat_engineering",
    "data_expectations",
    "data_drift",
    "data_split",
    "model_train",
    "model_train_challenger",
    "model_predict",
    "model_predict_challenger",
    "model_explainability",
    "__default__",
]
print("Project root:", PROJECT_ROOT)
print("Ticker:", TICKER)
print("Official price column:", FEATURE_PRICE_COLUMN)
print("Model features:", MARKET_FEATURE_COLUMNS)
print("Registered Kedro pipelines:", pipeline_names)


## 3. Data Ingestion from Yahoo Finance

The primary ingestion path downloads S&P 500 index data from Yahoo Finance using `yfinance` and ticker `^GSPC`. A local CSV fallback is allowed only to keep the notebook usable when the network or Yahoo endpoint is temporarily unavailable.


In [ ]:
def normalize_yfinance_download(downloaded: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Normalize yfinance output to the project's expected OHLCV schema."""
    if downloaded.empty:
        raise ValueError(f"Yahoo Finance returned no rows for {ticker}.")

    data = downloaded.copy()

    if isinstance(data.columns, pd.MultiIndex):
        expected = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}
        level_values = [set(map(str, data.columns.get_level_values(i))) for i in range(data.columns.nlevels)]

        if expected.intersection(level_values[0]):
            if data.columns.nlevels > 1 and ticker in level_values[1]:
                data = data.xs(ticker, axis=1, level=1)
            else:
                data.columns = data.columns.get_level_values(0)
        elif data.columns.nlevels > 1 and expected.intersection(level_values[1]):
            if ticker in level_values[0]:
                data = data.xs(ticker, axis=1, level=0)
            else:
                data.columns = data.columns.get_level_values(1)
        else:
            raise ValueError(f"Unexpected multi-index columns from yfinance: {data.columns}")

    data = data.reset_index()
    if "Datetime" in data.columns and "Date" not in data.columns:
        data = data.rename(columns={"Datetime": "Date"})

    if "Date" not in data.columns:
        raise ValueError("Downloaded data does not contain a Date column after normalization.")

    data["Date"] = pd.to_datetime(data["Date"], errors="raise").dt.tz_localize(None)

    missing_columns = [column for column in REQUIRED_COLUMNS if column not in data.columns]
    if missing_columns:
        raise ValueError(f"Downloaded data is missing required columns: {missing_columns}")

    normalized = data.loc[:, REQUIRED_COLUMNS].copy()
    normalized = normalized.sort_values("Date").reset_index(drop=True)
    return normalized


def download_sp500_market_data() -> pd.DataFrame:
    """Download S&P 500 market data, with an explicit cached fallback for offline runs."""
    download_error = None
    if DOWNLOAD_FROM_YAHOO:
        try:
            downloaded = yf.download(TICKER, start=START_DATE, progress=False, auto_adjust=False, group_by="column", threads=False)
            raw = normalize_yfinance_download(downloaded, TICKER)
            print("Ingestion source: Yahoo Finance live download")
            return raw
        except Exception as exc:
            download_error = exc
            print(f"Yahoo Finance download failed: {exc}")

    if ALLOW_CACHED_RAW_FALLBACK and RAW_DATA_PATH.exists():
        cached = pd.read_csv(RAW_DATA_PATH)
        cached["Date"] = pd.to_datetime(cached["Date"], errors="raise")
        print("Ingestion source: cached local CSV")
        return cached

    raise RuntimeError("Could not obtain S&P 500 data from Yahoo Finance and no cached CSV is available.") from download_error


raw_data = download_sp500_market_data()
validated_for_save = validate_raw_market_data(raw_data)
validated_for_save.to_csv(RAW_DATA_PATH, index=False)

print(f"Saved raw CSV to: {RAW_DATA_PATH}")
print("Rows:", len(validated_for_save))
print("Date range:", validated_for_save["Date"].min().date(), "to", validated_for_save["Date"].max().date())
print("Columns:", validated_for_save.columns.tolist())
display(validated_for_save.head())


## 4. Raw Data Quality Gate

This stage mirrors the `data_quality` pipeline: required OHLCV columns, parseable unique dates, positive prices, non-negative volume and basic OHLC relationships.


In [ ]:
print("Raw dataset shape:", raw_data.shape)
display(raw_data.dtypes.to_frame("dtype"))
display(raw_data.isna().sum().to_frame("missing_values"))

duplicate_date_count = raw_data["Date"].duplicated().sum()
print("Duplicate-date count:", duplicate_date_count)
print("Raw date range:", raw_data["Date"].min().date(), "to", raw_data["Date"].max().date())

ohlcv_checks = {
    "all_required_columns_present": set(REQUIRED_COLUMNS).issubset(raw_data.columns),
    "prices_strictly_positive": bool((raw_data[["Open", "High", "Low", "Close", "Adj Close"]] > 0).all().all()),
    "volume_non_negative": bool((raw_data["Volume"] >= 0).all()),
    "high_at_least_low": bool((raw_data["High"] >= raw_data["Low"]).all()),
    "high_at_least_open": bool((raw_data["High"] >= raw_data["Open"]).all()),
    "high_at_least_close": bool((raw_data["High"] >= raw_data["Close"]).all()),
    "open_at_least_low": bool((raw_data["Open"] >= raw_data["Low"]).all()),
    "close_at_least_low": bool((raw_data["Close"] >= raw_data["Low"]).all()),
}
display(pd.Series(ohlcv_checks, name="passed").to_frame())

validated_data = validate_raw_market_data(raw_data)
print("Raw quality gate: PASS")


## 5. Data Cleaning

The cleaning stage standardises the raw market data before feature engineering. The resulting CSV is persisted as an intermediate artefact so downstream stages have a stable input.


In [ ]:
cleaned_data = clean_raw_market_data(validated_data)
cleaned_data["Date"] = pd.to_datetime(cleaned_data["Date"], errors="raise")

assert cleaned_data["Date"].is_monotonic_increasing, "Cleaned data must be sorted by Date."
assert not cleaned_data["Date"].duplicated().any(), "Cleaned data must not contain duplicate dates."

cleaned_data.to_csv(CLEAN_DATA_PATH, index=False)

print("Cleaned dataset shape:", cleaned_data.shape)
print("Cleaned date range:", cleaned_data["Date"].min().date(), "to", cleaned_data["Date"].max().date())
print(f"Saved cleaned CSV to: {CLEAN_DATA_PATH}")
display(cleaned_data.head())


## 6. Exploratory Data Analysis

The EDA is intentionally compact: enough to confirm price history, return behaviour and target balance without turning this support notebook into a research report.


In [ ]:
eda_data = cleaned_data.copy()
eda_data["daily_return"] = eda_data["Close"].pct_change()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(eda_data["Date"], eda_data["Close"], linewidth=1.2)
axes[0].set_title("S&P 500 Close Price")
axes[0].set_ylabel("Close")
axes[0].grid(alpha=0.25)

axes[1].plot(eda_data["Date"], eda_data["daily_return"], linewidth=0.8, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Daily Returns")
axes[1].set_ylabel("Return")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

display(eda_data[["Open", "High", "Low", "Close", "Adj Close", "Volume", "daily_return"]].describe().T)


## 7. Feature Engineering and Label Construction

This stage uses the project source function `create_market_feature_dataset`. Features are computed from `Close`, while `Adj Close` is retained only as raw source information.


In [ ]:
feature_data = create_market_feature_dataset(cleaned_data)
feature_data["Date"] = pd.to_datetime(feature_data["Date"], errors="raise")
feature_data.to_csv(FEATURE_DATA_PATH, index=False)

print("Feature dataset shape:", feature_data.shape)
print("Feature date range:", feature_data["Date"].min().date(), "to", feature_data["Date"].max().date())
print("Official price column used by the project:", FEATURE_PRICE_COLUMN)
print("Feature columns:", MARKET_FEATURE_COLUMNS)
print("Target column:", TARGET_COLUMN)
print(f"Saved feature CSV to: {FEATURE_DATA_PATH}")

display(feature_data[["Date", *MARKET_FEATURE_COLUMNS, TARGET_COLUMN]].head())
display(feature_data[TARGET_COLUMN].value_counts().rename("count").to_frame())
display(feature_data[TARGET_COLUMN].value_counts(normalize=True).rename("share").to_frame())


## 8. Data Contracts with Great Expectations

This stage creates compact validation artefacts for both the raw dataset and the engineered feature dataset. These reports are useful evidence for the MLOps project because they make quality gates explicit and reproducible.


In [ ]:
raw_validation_result = validate_raw_market_data_expectations(raw_data)
feature_validation_result = validate_feature_data_expectations(feature_data)

raw_validation_report = create_raw_validation_report(raw_validation_result)
feature_validation_report = create_feature_validation_report(feature_validation_result)
raw_validation_html = create_raw_validation_html_report(raw_validation_report)
feature_validation_html = create_feature_validation_html_report(feature_validation_report)

raw_report_path = EXPECTATIONS_DIR / "raw_data_validation_report.json"
feature_report_path = EXPECTATIONS_DIR / "feature_data_validation_report.json"
raw_html_path = EXPECTATIONS_DIR / "raw_data_validation_report.html"
feature_html_path = EXPECTATIONS_DIR / "feature_data_validation_report.html"

raw_report_path.write_text(json.dumps(raw_validation_report, indent=2), encoding="utf-8")
feature_report_path.write_text(json.dumps(feature_validation_report, indent=2), encoding="utf-8")
raw_html_path.write_text(raw_validation_html, encoding="utf-8")
feature_html_path.write_text(feature_validation_html, encoding="utf-8")

enforce_validation_contracts(raw_validation_report, feature_validation_report)

validation_summary = pd.DataFrame([raw_validation_report, feature_validation_report])[
    ["dataset_name", "validation_name", "success", "total_expectations", "failed_expectations", "validation_timestamp"]
]
display(validation_summary)
print("Validation artefacts saved to:", EXPECTATIONS_DIR)


## 9. Data Drift Monitoring Snapshot

The drift stage compares the oldest 70 percent of engineered feature rows with the most recent 30 percent. This is not a production monitor by itself, but it produces a reproducible drift snapshot suitable for MLOps evidence.


In [ ]:
reference_data, current_data = split_reference_current_data(feature_data)
reference_features, current_features = select_numeric_drift_features(reference_data, current_data)

drift_html, drift_summary = create_data_drift_report(reference_features, current_features)
drift_html_path = DRIFT_DIR / "data_drift_report.html"
drift_summary_path = DRIFT_DIR / "data_drift_summary.json"
drift_html_path.write_text(drift_html, encoding="utf-8")
drift_summary_path.write_text(json.dumps(drift_summary, indent=2), encoding="utf-8")

drift_overview = pd.DataFrame([
    {
        "reference_rows": drift_summary["reference_rows"],
        "current_rows": drift_summary["current_rows"],
        "dataset_drift": drift_summary["dataset_drift"],
        "drifted_columns_count": drift_summary["drifted_columns_count"],
        "drifted_columns_share": drift_summary["drifted_columns_share"],
    }
])
display(drift_overview)
print("Drift artefacts saved to:", DRIFT_DIR)


## 10. Explicit Temporal Train / Validation / Test Split

The notebook uses calendar boundaries to make the split easy to explain in the final project. Training covers the long historical period, validation covers 2024-2025, and test covers 2026 onward when available from Yahoo Finance.


In [ ]:
def split_by_calendar_boundaries(feature_dataset: pd.DataFrame) -> dict[str, pd.DataFrame | pd.Series]:
    data = feature_dataset.copy()
    data["Date"] = pd.to_datetime(data["Date"], errors="raise")
    data = data.sort_values("Date").reset_index(drop=True)

    train = data[data["Date"] <= pd.Timestamp(TRAIN_END)].copy()
    validation = data[(data["Date"] >= pd.Timestamp(VALIDATION_START)) & (data["Date"] <= pd.Timestamp(VALIDATION_END))].copy()
    test = data[data["Date"] >= pd.Timestamp(TEST_START)].copy()

    if train.empty:
        raise ValueError("Training split is empty. Check START_DATE and TRAIN_END.")
    if validation.empty:
        raise ValueError("Validation split is empty. Check VALIDATION_START and VALIDATION_END.")
    if test.empty:
        raise ValueError("Test split is empty. The latest available Yahoo data may not include 2026 yet.")

    assert train["Date"].max() < validation["Date"].min(), "Train and validation periods overlap."
    assert validation["Date"].max() < test["Date"].min(), "Validation and test periods overlap."
    assert data["Date"].is_monotonic_increasing, "Feature data must be chronologically sorted."

    return {
        "train": train,
        "validation": validation,
        "test": test,
        "X_train": train.loc[:, MARKET_FEATURE_COLUMNS].copy(),
        "y_train": train[TARGET_COLUMN].copy(),
        "X_val": validation.loc[:, MARKET_FEATURE_COLUMNS].copy(),
        "y_val": validation[TARGET_COLUMN].copy(),
        "X_test": test.loc[:, MARKET_FEATURE_COLUMNS].copy(),
        "y_test": test[TARGET_COLUMN].copy(),
    }

splits = split_by_calendar_boundaries(feature_data)

split_summary = []
for split_name in ["train", "validation", "test"]:
    split_df = splits[split_name]
    split_summary.append({
        "split": split_name,
        "rows": len(split_df),
        "first_date": split_df["Date"].min().date(),
        "last_date": split_df["Date"].max().date(),
        "target_0": int((split_df[TARGET_COLUMN] == 0).sum()),
        "target_1": int((split_df[TARGET_COLUMN] == 1).sum()),
    })

split_summary = pd.DataFrame(split_summary)
display(split_summary)

X_train = splits["X_train"]
y_train = splits["y_train"]
X_val = splits["X_val"]
y_val = splits["y_val"]
X_test = splits["X_test"]
y_test = splits["y_test"]

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)


## 11. Shared Model Evaluation Utilities

Both models are evaluated through the same functions so the comparison remains fair and repeatable.


In [ ]:
def compute_metrics(model, X: pd.DataFrame, y: pd.Series) -> dict[str, float]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else None
    metrics = {
        "accuracy": float(accuracy_score(y, predictions)),
        "precision": float(precision_score(y, predictions, zero_division=0)),
        "recall": float(recall_score(y, predictions, zero_division=0)),
        "f1_score": float(f1_score(y, predictions, zero_division=0)),
    }
    if probabilities is not None and y.nunique() == 2:
        metrics["roc_auc"] = float(roc_auc_score(y, probabilities))
    else:
        metrics["roc_auc"] = np.nan
    return metrics


def evaluate_model(model_name: str, model, datasets: dict[str, tuple[pd.DataFrame, pd.Series]]) -> pd.DataFrame:
    rows = []
    for split_name, (X, y) in datasets.items():
        rows.append({"model": model_name, "split": split_name, **compute_metrics(model, X, y)})
    return pd.DataFrame(rows)


def show_confusion_matrix(model, X: pd.DataFrame, y: pd.Series, title: str) -> None:
    predictions = model.predict(X)
    cm = confusion_matrix(y, predictions, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["down", "up"])
    disp.plot(values_format="d", cmap="Blues")
    plt.title(title)
    plt.grid(False)
    plt.show()


datasets = {"train": (X_train, y_train), "validation": (X_val, y_val), "test": (X_test, y_test)}


## 12. Baseline Model: Logistic Regression

Logistic Regression is the baseline because it is fast, stable and interpretable. The scaler is part of the sklearn pipeline to keep training and inference transformations consistent.


In [ ]:
lr_model = Pipeline(steps=[("scaler", StandardScaler()), ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))])
lr_model.fit(X_train, y_train)

lr_metrics = evaluate_model("logistic_regression", lr_model, datasets)
display(lr_metrics)

for split_name, (X, y) in {"validation": (X_val, y_val), "test": (X_test, y_test)}.items():
    print(f"Logistic Regression classification report - {split_name}")
    print(classification_report(y, lr_model.predict(X), zero_division=0))
    show_confusion_matrix(lr_model, X, y, f"Logistic Regression Confusion Matrix - {split_name}")


## 13. Challenger Model: Random Forest

Random Forest is the challenger because it can capture nonlinear interactions in the engineered market features. Its role here is comparative support for the MLOps workflow, not proof of a superior trading model.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_metrics = evaluate_model("random_forest", rf_model, datasets)
display(rf_metrics)

for split_name, (X, y) in {"validation": (X_val, y_val), "test": (X_test, y_test)}.items():
    print(f"Random Forest classification report - {split_name}")
    print(classification_report(y, rf_model.predict(X), zero_division=0))
    show_confusion_matrix(rf_model, X, y, f"Random Forest Confusion Matrix - {split_name}")

feature_importance = pd.DataFrame({"feature": MARKET_FEATURE_COLUMNS, "importance": rf_model.feature_importances_}).sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feature_importance["feature"], feature_importance["importance"])
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
ax.grid(axis="x", alpha=0.25)
plt.show()


## 14. Model Selection on Validation Set

The candidate champion is selected using validation `f1_score`, with `roc_auc` as the tie-breaker. This keeps the test split untouched until the final evaluation stage.


In [ ]:
all_metrics = pd.concat([lr_metrics, rf_metrics], ignore_index=True)
validation_comparison = all_metrics[all_metrics["split"] == "validation"].sort_values(["f1_score", "roc_auc"], ascending=False).reset_index(drop=True)
display(validation_comparison)

best_model_name = validation_comparison.loc[0, "model"]
candidate_champion = lr_model if best_model_name == "logistic_regression" else rf_model

print("Candidate champion selected on validation set:", best_model_name)
if validation_comparison.loc[0, "roc_auc"] <= 0.55:
    print("Warning: validation ROC-AUC is close to random discrimination. Interpret predictive value cautiously.")


## 15. Final Test Evaluation

Only after model selection do we report the chosen model's held-out test performance. The test period is necessarily short when the notebook is run early in 2026, so the numbers should be presented honestly and cautiously.


In [ ]:
selected_test_metrics = pd.DataFrame([{"model": best_model_name, "split": "test", **compute_metrics(candidate_champion, X_test, y_test)}])
display(selected_test_metrics)

print(f"Candidate champion classification report - test ({best_model_name})")
print(classification_report(y_test, candidate_champion.predict(X_test), zero_division=0))
show_confusion_matrix(candidate_champion, X_test, y_test, f"Candidate Champion Test Confusion Matrix - {best_model_name}")

test_start = splits["test"]["Date"].min().date()
test_end = splits["test"]["Date"].max().date()
print(f"Final test period: {test_start} to {test_end}")
print("Limitations: financial markets are noisy and non-stationary; the test period is limited; class balance can change over time; ROC-AUC near 0.50 should be reported honestly.")


## 16. Persist Model, Metrics and Prediction Artefacts

This stage writes the support artefacts that are useful for the MLOps project: trained models, feature names, metrics and test predictions. The project can still use MLflow elsewhere; these local files make the notebook self-contained.


In [ ]:
def save_pickle(obj, path: Path) -> None:
    with path.open("wb") as file:
        pickle.dump(obj, file)

lr_model_path = MODEL_DIR / "logistic_regression_support_model.pkl"
rf_model_path = MODEL_DIR / "random_forest_support_model.pkl"
champion_model_path = MODEL_DIR / "candidate_champion_support_model.pkl"
metrics_path = MODEL_OUTPUT_DIR / "support_model_metrics.csv"
selection_path = MODEL_OUTPUT_DIR / "support_model_selection.json"
test_predictions_path = MODEL_OUTPUT_DIR / "candidate_champion_test_predictions.csv"

save_pickle(lr_model, lr_model_path)
save_pickle(rf_model, rf_model_path)
save_pickle(candidate_champion, champion_model_path)
all_metrics.to_csv(metrics_path, index=False)

selection_payload = {
    "selected_model": best_model_name,
    "selection_split": "validation",
    "selection_metric": "f1_score",
    "tie_breaker_metric": "roc_auc",
    "feature_columns": MARKET_FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
selection_path.write_text(json.dumps(selection_payload, indent=2), encoding="utf-8")

test_predictions = splits["test"][["Date", TARGET_COLUMN]].copy()
test_predictions["predicted_target"] = candidate_champion.predict(X_test).astype(int)
test_predictions["probability_up"] = candidate_champion.predict_proba(X_test)[:, 1]
test_predictions.to_csv(test_predictions_path, index=False)

artifact_table = pd.DataFrame([
    {"artifact": "logistic_regression_model", "path": str(lr_model_path)},
    {"artifact": "random_forest_model", "path": str(rf_model_path)},
    {"artifact": "candidate_champion_model", "path": str(champion_model_path)},
    {"artifact": "all_metrics", "path": str(metrics_path)},
    {"artifact": "model_selection", "path": str(selection_path)},
    {"artifact": "test_predictions", "path": str(test_predictions_path)},
])
display(artifact_table)


## 17. Explainability Artefacts

For Logistic Regression, the scaled coefficients provide a compact directional interpretation. For Random Forest, impurity-based importances provide a simple challenger explanation. These artefacts are lightweight, deterministic and easy to include in the project report.


In [ ]:
lr_classifier = lr_model.named_steps["classifier"]
lr_importance = pd.DataFrame({
    "feature": MARKET_FEATURE_COLUMNS,
    "scaled_coefficient": lr_classifier.coef_[0],
    "absolute_scaled_coefficient": np.abs(lr_classifier.coef_[0]),
}).sort_values("absolute_scaled_coefficient", ascending=False)

rf_importance = pd.DataFrame({"feature": MARKET_FEATURE_COLUMNS, "importance": rf_model.feature_importances_}).sort_values("importance", ascending=False)

lr_importance_path = REPORTING_DIR / "logistic_regression_support_coefficients.csv"
rf_importance_path = REPORTING_DIR / "random_forest_support_feature_importance.csv"
lr_importance.to_csv(lr_importance_path, index=False)
rf_importance.to_csv(rf_importance_path, index=False)

display(lr_importance)
display(rf_importance)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lr_plot = lr_importance.sort_values("scaled_coefficient")
axes[0].barh(lr_plot["feature"], lr_plot["scaled_coefficient"])
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Logistic Regression Scaled Coefficients")
axes[0].grid(axis="x", alpha=0.25)

rf_plot = rf_importance.sort_values("importance")
axes[1].barh(rf_plot["feature"], rf_plot["importance"])
axes[1].set_title("Random Forest Feature Importance")
axes[1].grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

print("Explainability artefacts saved to:", REPORTING_DIR)


## 18. Serving-Style Prediction Example

This cell uses the most recent engineered row as a single-record inference payload. It demonstrates the same feature contract that an API or batch scorer would need to satisfy.


In [ ]:
latest_row = feature_data.sort_values("Date").iloc[-1]
latest_features = latest_row[MARKET_FEATURE_COLUMNS].to_frame().T
latest_prediction = int(candidate_champion.predict(latest_features)[0])
latest_probability_up = float(candidate_champion.predict_proba(latest_features)[:, 1][0])

serving_payload = latest_features.iloc[0].to_dict()
prediction_summary = pd.DataFrame([
    {
        "date": latest_row["Date"].date(),
        "model": best_model_name,
        "predicted_class": latest_prediction,
        "predicted_label": "up" if latest_prediction == 1 else "down",
        "probability_up": latest_probability_up,
    }
])

payload_path = MODEL_OUTPUT_DIR / "latest_serving_payload.json"
prediction_path = MODEL_OUTPUT_DIR / "latest_prediction_summary.json"
payload_path.write_text(json.dumps(serving_payload, indent=2), encoding="utf-8")
prediction_path.write_text(prediction_summary.iloc[0].to_json(indent=2), encoding="utf-8")

display(latest_features)
display(prediction_summary)
print("Serving payload saved to:", payload_path)
print("Prediction summary saved to:", prediction_path)
print("This prediction is a technical demonstration only and is not financial advice.")


## 19. Final Summary

This notebook now implements the complete support workflow for the MLOps project:

- downloads S&P 500 data from Yahoo Finance using `^GSPC`;
- validates raw data quality and explicit data contracts;
- cleans and feature-engineers the dataset using the project source code;
- produces drift evidence for the engineered features;
- trains Logistic Regression and Random Forest models;
- selects a candidate champion on validation data;
- reports final test metrics only after model selection;
- persists local artefacts for models, metrics, predictions, data contracts, drift and explainability;
- demonstrates a serving-style prediction payload.

All modelling results should be framed as MLOps evidence rather than as financial advice or as a deployable trading edge.
